# 06 — Structured Data

**Objectives:**
- Extract clean structured data (JSON) from Claude responses
- Use message prefilling and stop sequences to control output format

**Exam domain:** Context Management & Reliability (15%)

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
# Get the client and model from utils
from src.utils import get_client, get_model, chat, add_user_message, add_assistant_message
from IPython.display import Markdown
import json

client = get_client()
model = get_model()

## 1. Structured Data

It is common to generate pieces of structured data.

In [3]:
# Claude uses to format the response to be printable in a markdown code block, so we have to remove the code block formatting to get the raw json
messages = []

add_user_message(
    messages,
    "Generate a very short event bridge rule as json"
)

chat(messages, client, model, temperature=0.0)

'```json\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n```'

In [4]:
messages = []

add_user_message(
    messages,
    "Generate a very short event bridge rule as json"
)

# Already do the ```json part, so the model just has to fill in the rest of the json without adding any extra text
add_assistant_message(
    messages,
    "```json"
)

response = chat(messages, client, model, temperature=0.0, stop_sequences=["```"])
print(response)


{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}



In [5]:
# Now the answer is completely legible by json parsers
json.loads(response)

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

## 2. Exercise — Message Prefilling and Stop Sequences

- Use message prefilling and stop sequences only to get three different commands in a single response
- There shouldn't be any comments or explanation
- Hint: message prefilling isn't limited to just characters like ` ``` `

In [6]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

text = chat(messages, client, model, temperature=0.0)

text

'Here are three short AWS CLI commands:\n\n1. **List S3 buckets:**\n   ```bash\n   aws s3 ls\n   ```\n\n2. **Describe EC2 instances:**\n   ```bash\n   aws ec2 describe-instances\n   ```\n\n3. **Get caller identity:**\n   ```bash\n   aws sts get-caller-identity\n   ```'

In [7]:
Markdown(text)

Here are three short AWS CLI commands:

1. **List S3 buckets:**
   ```bash
   aws s3 ls
   ```

2. **Describe EC2 instances:**
   ```bash
   aws ec2 describe-instances
   ```

3. **Get caller identity:**
   ```bash
   aws sts get-caller-identity
   ```

In [8]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, client, model, temperature=0.0, stop_sequences=["```"])

text.strip()

'aws s3 ls\naws ec2 describe-instances\naws iam list-users'